In [11]:
import cv2
import glob
import numpy as np
import re
import easyocr
from ultralytics import YOLO

# ---------------------------
# 1. Предобработка (только CLAHE, без бинаризации)
# ---------------------------
def preprocess_plate(plate_img):
    img = plate_img.copy()
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return img

# ---------------------------
# 2. Мягкая валидация (возвращает очищенный текст, даже если маска не подошла)
# ---------------------------
def validate_and_correct_plate(text):
    # Оставляем только буквы и цифры
    clean = re.sub(r'[^A-ZА-Яa-zа-я0-9]', '', text).upper()
    if not clean:
        return None
    
    # Попытка привести к формату российского номера
    letters = 'АВЕКМНОРСТУХABEKMHOPCTYX'
    digits = '0123456789'
    pattern = r'^([' + letters + r'])([' + digits + r']{3})([' + letters + r']{2})([' + digits + r']{2,3})$'
    match = re.match(pattern, clean)
    if match:
        return f"{match.group(1)}{match.group(2)}{match.group(3)}{match.group(4)}"
    
    # Если не подошло — возвращаем очищенный текст (без замен)
    return clean

# ---------------------------
# 3. Основная функция распознавания
# ---------------------------
def recognize_plate(plate_img, reader, debug=False):
    # Предобработка
    processed = preprocess_plate(plate_img)
    
    # Разрешаем все латинские буквы и цифры (и кириллицу)
    allowed_chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789АВЕКМНОРСТУХ'
    
    # EasyOCR с ограничениями
    result = reader.readtext(
        processed,
        allowlist=allowed_chars,
        paragraph=False,
        width_ths=0.7,
        height_ths=0.7,
        x_ths=0.3,
        y_ths=0.3,
        detail=1
    )
    
    if not result:
        # Пробуем без allowlist (на случай, если символы не в списке)
        result = reader.readtext(processed, paragraph=False, detail=1)
        if not result:
            if debug:
                print("OCR не вернул ни одного символа.")
            return None
    
    raw_text = result[0][1]
    confidence = result[0][2] if len(result[0]) > 2 else 0.0
    
    if debug:
        print(f"Raw OCR: '{raw_text}' (conf={confidence:.2f})")
    
    # Валидация + коррекция (мягкая)
    validated = validate_and_correct_plate(raw_text)
    
    if debug:
        if validated:
            print(f"After validation: '{validated}'")
        else:
            print("Validation failed, returning raw text.")
    
    # Если валидация вернула None — используем сырой текст
    return validated if validated is not None else raw_text

# ---------------------------
# 4. Основной пайплайн
# ---------------------------
model = YOLO(r"runs/detect/train-3/weights/best.pt")
reader = easyocr.Reader(['en'], gpu=True)   # можно добавить 'ru'

images = glob.glob(r"dataset/test/images/*.jpg")
print("IMAGES:", len(images))

for img_path in images:
    img = cv2.imread(img_path)
    results = model.predict(img, conf=0.25, verbose=False)
    
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = img[y1:y2, x1:x2]
            
            plate_number = recognize_plate(crop, reader, debug=False)   # debug=True для отладки
            text = plate_number if plate_number else ""
            print(f"{img_path} → {text}")

IMAGES: 63
dataset/test/images\09-19-06-Cam1_jpg.rf.a794b758a807ec6ac08b6d2f82e0f175.jpg → AC
dataset/test/images\09-19-06-Cam2_jpg.rf.2110d0ac45022f2e9062bd320e03c1e1.jpg → 136
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → EO9OBAI164
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → I4E
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → 
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → 
dataset/test/images\10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg → 
dataset/test/images\11_6_2014_18_42_24_899_png.rf.8497e3135b77e92d28c28baea2923e5c.jpg → 09991049
dataset/test/images\11_jpg.rf.90f13ef3216a7240666417476e8aca65.jpg → MZOOMMEE
dataset/test/images\12_6_2014_19_54_57_849_png.rf.5f74991818bfb9bcb5b7ae9755b1b96d.jpg → EM186K15
dataset/test/images\15_jpg.rf.dcc03c3b96729ac3b36e69aa3053a197.jpg → X57SEPI15O
dataset/test/images\20230405-112422_jpg.rf.69856ec911d503a58c6ad8db5c6a7537.jpg → L0303TEF5
dataset/test/